# Plan summary：
## 1. Base Model
- DeepSeek-R1-Distill-Qwen-14B
- NuminaMath-CoT 7B (可行性验证)
- NuminaMath-TIR 7B (可行性验证)

## 2. Datasets
- NuminaMath-CoT，用于强化思维链
- NuminaMath-TIR，用于强化python 工具调用，训练数据格式为 ： 推理块，python块，output块 的组合

## 3. 监督微调
- 用 NuminaMath-CoT 微调 DeepSeek-R1-Distill-Qwen-32B 得到 model_cot
- 用 NuminaMath-TIR 微调 DeepSeek-R1-Distill-Qwen-32B 得到 model_tir

## 4. Inference
- 构造 Chain of Thought prompt 和 Tool Integrated Reasoning prompt，分别输入 model_cot, model_tir
- 对 model_tir 的输出，重复以下循环直到输出 \box{} 答案：
  - 尝试捕获 python 块
  - 捕获后立刻终止输出
  - 调用python得到结果
  - 包装为 output 块，追加到原输出的 python块后面
  - 将追加后的完整输出输入原模型
- 并行进行16组输出
- 重复上述并行输出4轮，第i轮输入包含第1,...,i-1轮的输入，模型输出（追加了python块执行结果），答案正确与否
- 汇总答案，输出得票最多的结果

## 5. Acceleration
- vllm
- list_of_messages 批处理输入

## 6. Extensions
- 训练奖励模型用于给模型的完整回答评分，得到各答案的加权
- AlphaGeometry 
- AlphaProof

#### 参考
- DeepMind AlphaProof
- DeepMind AlphaGeometry
- NuminaMath : https://www.kaggle.com/code/lewtun/numina-1st-place-solution
- Imagination Research AIMO 2

# 版本：
#### v2
- 模型：
    - NuminaMath-CoT 7B 
    - NuminaMath-TIR 7B 
- cot, tir 双提示词 -> 对应模型
- vllm batched input & multiple sampled outputs
- python工具调用
- 简单投票


#### v3
- vllm加载


#### v？
- 改进答案提取
- 答案基本属性检查
- batched思维链生成




# **注：**
1. 怎么生成多条思维链：
   - 不同 generation config
   - 在关键步骤分支
   - Tree of thought : BFS。对思考质量高的分支继续探索

2. 怎么选择最终的思维链：
   - 选出得票最多的答案（加权：推理长度，自洽性，答案格式）
   - 分析不同路径的推理质量
   - 多智能体答辩
   

In [17]:
import os
import re
import pandas as pd
from tqdm import tqdm

import torch
from modelscope import snapshot_download, AutoModelForCausalLM, AutoTokenizer

In [18]:
# 1. system
REF_CSV_PATH = "../AIMO/reference.csv"
TEST_CSV_PATH = "../AIMO/test.csv"
OUTPUT_CSV_PATH = "../Solutions/solutions_deepseek.csv"
MODEL_ID = "deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B" 
MODEL_DIR = "../Models"

os.makedirs(MODEL_DIR, exist_ok=True)


# 2. sampling params
# DeepSeek-R1 推荐的生成参数
GENERATION_CONFIG = {
    "do_sample": True,
    "temperature": 0.6,           # R1通常用稍低的温度
    "top_p": 0.95,
    "top_k": 50,                   # R1推荐使用top_k
    "repetition_penalty": 1.1,     # 稍微提高防止重复
    "max_new_tokens": 20000,         # R1可能需要更多token（因为推理步骤更长）
    "min_length": 100,              # 确保有足够长的推理
}


# 3. prompts
# 3.1. System Instruction
SYSTEM_PROMPT = (
    "You are a helpful mathematics expert. "
    "You will solve math problems by reasoning step by step. "
    "Put your final answer within \\boxed{}."
)

# 3.2 Few shot examples
# R1系列不需要few-shot，它有自己的推理格式


In [2]:
SYSTEM_PROMPT_COT = (
    "You are a helpful mathematics expert. "
    "You will solve math problems by reasoning step by step. "
    "Put your final answer within \\boxed{}."
)
SYSTEM_PROMPT_COT = [SYSTEM_PROMPT_COT,] * 4
print(type(SYSTEM_PROMPT_COT))

<class 'list'>


## Part 1 加载模型

In [ ]:
if torch.cuda.is_available():
    device = "cuda"
    print(f"使用 CUDA: {torch.cuda.get_device_name(0)}")
    # 如果有多个 GPU，可以使用 device_map="auto" 让 accelerate 自动分配
    # device_map = "auto" 
    dtype = torch.bfloat16 # NVIDIA Ampere+ 支持良好
    use_flash_attn = True
elif torch.backends.mps.is_available():
    device = "mps"
    print("使用 MPS")
    dtype = torch.float16 # MPS 上 float16 通常比 bfloat16 更稳定
    use_flash_attn = False # MPS 不支持 Flash Attn 2
else:
    device = "cpu"
    print("使用 CPU")
    dtype = torch.float32
    use_flash_attn = False


[1/4] 加载DeepSeek-R1-1.5B模型...
使用 GPU


In [ ]:
# Step 1.1 下载模型
model_path_cot = snapshot_download(     # 如果目录已存在且完整，它会秒级返回，不会重复下载
    MODEL_ID_COT, 
    cache_dir=MODEL_DIR, 
    revision='master'  # 可选：指定版本，如 'v1.0.0'
)

model_path_tir = snapshot_download(     # 如果目录已存在且完整，它会秒级返回，不会重复下载
    MODEL_ID_COT, 
    cache_dir=MODEL_DIR, 
    revision='master'  # 可选：指定版本，如 'v1.0.0'
)

2026-03-16 18:03:31,656 - modelscope - INFO - Target directory already exists, skipping creation.


In [21]:
# Step 1.2 加载model和tokenizer

model = AutoModelForCausalLM.from_pretrained(
    model_path, 
    device_map=device, 
    trust_remote_code=True,
    dtype=torch.bfloat16, 
)

tokenizer = AutoTokenizer.from_pretrained(model_path, trust_remote_code=True)
if tokenizer.pad_token is None:         # 设置padding token（如果未设置）
    tokenizer.pad_token = tokenizer.eos_token
    
print(f"模型加载成功！设备：{model.device}, 数据类型：{model.dtype}")


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

模型加载成功！设备：mps:0, 数据类型：torch.bfloat16


In [22]:
# # 简单测试
# prompt = "你好，请介绍一下你自己。"
# inputs = tokenizer(prompt, return_tensors="pt").to(device)
# outputs = model.generate(**inputs, max_new_tokens=50)
# response = tokenizer.decode(outputs[0], skip_special_tokens=True)
# print(response)

## Part 2 数据加载

In [23]:
def load_questions(path):
    df = pd.read_csv(path)
    ids = df["id"].tolist()
    problems = df["problem"].tolist()
    answers = None
    if "answer" in df.columns:
        answers = df["answer"].tolist()
        
    return ids, problems, answers


In [24]:
print("\n[2/4] 加载测试数据...")
test_ids, test_questions, test_answers = load_questions(TEST_CSV_PATH)


[2/4] 加载测试数据...


## Part 3 主推理函数

In [25]:
def build_message(question: str):
    messages = [{
        "role" : "user",
        "content" : f"{SYSTEM_PROMPT}"
    }]
    messages.append(
        {"role": "user", "content": f"Please solve this math problem:\n{question}\nLet's think step by step."}
    )
    return messages

In [26]:
def extract_deepseek_answer(text : str):
    # 标准boxed格式: \boxed{123}
    boxed_pattern = re.compile(r'\\boxed\s*\{([^}]*)\}')
    match = boxed_pattern.search(text)
    if match:
        return match.group(1).strip()
    
    # 2. 中文格式：答案是123
    chinese_pattern = re.compile(r"答案是[：:]\s*(\d+)")
    match = chinese_pattern.search(text)
    if match:
        return match.group(1)
    
    # 3. 英文格式：answer is 123
    english_pattern = re.compile(r"answer is[：:]\s*(\d+)", re.IGNORECASE)
    match = english_pattern.search(text)
    if match:
        return match.group(1)
    
    # 4. 最后一行可能是数字
    lines = text.strip().split('\n')
    for line in reversed(lines):
        numbers = re.findall(r'\b(\d+)\b', line)
        if numbers:
            return numbers[-1]
    
    return "Not Found"

In [27]:
def solve_single_problem(question: str, question_id: str = None):
    """
    使用DeepSeek-R1解决单个数学问题
    """
    try:
        # 1. 构建消息
        messages = build_message(question)
        
        # 2. 应用对话模板
        text_input = tokenizer.apply_chat_template(
            messages, 
            tokenize=False, 
            add_generation_prompt=True
        )
        
        # 3. 编码输入
        model_inputs = tokenizer([text_input], return_tensors="pt").to(model.device)
        input_length = model_inputs["input_ids"].shape[1]
        
        # 4. 生成推理
        with torch.no_grad():
            generated_ids = model.generate(
                **model_inputs,
                **GENERATION_CONFIG,
                pad_token_id=tokenizer.pad_token_id,
                eos_token_id=tokenizer.eos_token_id
            )
        
        # 5. 去除输入部分，只保留生成的回答
        generated_ids = generated_ids[:, input_length:]
        
        # 6. 解码
        response = tokenizer.batch_decode(
            generated_ids, 
            skip_special_tokens=True
        )[0].strip()
        
        # 7. 提取答案
        answer = extract_deepseek_answer(response)
        
        return response, answer
        
    except Exception as e:
        print(f"处理问题 {question_id} 时出错: {e}")
        return f"Error: {str(e)}", "Error"

In [ ]:
print("\n[3/4] 开始批量推理...")

results = []
start_time = pd.Timestamp.now()
last_finish_time = start_time

for idx, (q_id, question) in enumerate(tqdm(
    zip(test_ids, test_questions), 
    total=len(test_questions), 
    desc="推理进度"
)):
    
    # 1. 
    response, answer = solve_single_problem(question, q_id)
    
    # 2. 
    results.append({
        'id': q_id,
        'question': question,
        'full_response': response,
        'extracted_answer': answer
    })

    # 3. 日志
    cur_time = pd.Timestamp.now()
    elapse_from_last = (cur_time - last_finish_time).total_seconds()
    elapse_total = (cur_time - start_time).total_seconds()
    print(f"第{idx+1}题用时 : {elapse_from_last} \n 总计用时：{elapse_total} \n")




[3/4] 开始批量推理...


推理进度:  33%|███▎      | 1/3 [00:38<01:17, 38.68s/it]


  已处理 1/3 题, 平均 38.7秒/题, 预计剩余 1.3分钟


推理进度:  67%|██████▋   | 2/3 [02:07<01:07, 67.89s/it]


  已处理 2/3 题, 平均 63.5秒/题, 预计剩余 1.1分钟


推理进度: 100%|██████████| 3/3 [03:13<00:00, 64.41s/it]


  已处理 3/3 题, 平均 64.4秒/题, 预计剩余 0.0分钟


### Part 4 结果统计与保存

In [29]:
print("\n[4/4] 结果统计与保存:")

# 1. 生成提交文件
# 转换为DataFrame
result_df = pd.DataFrame(results)
submission = pd.DataFrame({
    'id': test_ids,
    'answer': result_df["extracted_answer"].tolist()
})

# 2. 保存
full_result_path = OUTPUT_CSV_PATH.replace('.csv', '_full.csv')
submission_path = OUTPUT_CSV_PATH.replace('.csv', '_submission.csv')
result_df.to_csv(full_result_path, index=False, encoding='utf-8-sig')
submission.to_csv(submission_path, index=False)

# 3. 运行时间统计
total_time = (pd.Timestamp.now() - start_time).total_seconds()
print(f"\n⏱️ 总运行时间: {total_time/60:.2f} 分钟")
print(f"⏱️ 平均每题: {total_time/len(test_questions):.2f} 秒")


[4/4] 结果统计与保存:

⏱️ 总运行时间: 3.23 分钟
⏱️ 平均每题: 64.53 秒
